# v0.14.0 -- Computed fields (`Computed[...]` -> `DEFINE FIELD ... VALUE`)

v0.13.0 let a **single write** compute a value on the server (`save(server_values=...)`).
v0.14.0 goes one step further and attaches the expression to the **schema**:

```python
full_name: Computed[str] = Computed("string::concat(first_name, ' ', last_name)")
```

Once `define_computed_fields()` has run, SurrealDB recomputes the field on *every* write to the
table -- `CREATE`, `UPDATE`, `MERGE`, `UPSERT`, `PATCH` -- no matter who issues it. That makes a
computed field a **server-enforced invariant**, not a convention your application code has to
remember.

This notebook walks the whole feature. Everything here behaves **identically on SurrealDB 2.6.x
and 3.x**, so there is no capability probe and no branching: the notebook runs top to bottom on
either line.

## 1. Connect

WebSocket (`.../rpc`). Point `SURREALDB_HOST` / `SURREALDB_PORT` at your own server to run this
notebook elsewhere.

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Declare a model with computed fields

`Computed[T]` is the annotation; `Computed("<expr>")` carries the SurrealQL expression. The
annotation resolves to `T | None` with a default of `None`, so you can build an instance before
the server has ever computed anything.

The expression may be a plain string or a `SurrealFunc` -- they share one validation path and one
trust model.

In [2]:
from surreal_orm_lite import BaseSurrealModel, Computed, SurrealFunc


class Order(BaseSurrealModel):
    id: str
    customer: str = ""
    unit_price: float = 0.0
    quantity: int = 0
    # Alphabetical order matters (see section 8): "subtotal" sorts before "total".
    subtotal: Computed[float] = Computed("unit_price * quantity")
    total: Computed[float] = Computed(SurrealFunc("math::round(subtotal * 1.15)"))
    label: Computed[str] = Computed("string::uppercase(customer)")


print("computed fields:", Order.get_computed_fields())

computed fields: {'subtotal': 'unit_price * quantity', 'total': 'math::round(subtotal * 1.15)', 'label': 'string::uppercase(customer)'}


## 3. Inspect the DDL before running it

`computed_field_ddl()` is pure -- it renders the statements without touching the database, so you
can print them, diff them, or feed them into a migration.

In [3]:
for statement in Order.computed_field_ddl():
    print(statement)

DEFINE FIELD OVERWRITE subtotal ON Order VALUE unit_price * quantity;
DEFINE FIELD OVERWRITE total ON Order VALUE math::round(subtotal * 1.15);
DEFINE FIELD OVERWRITE label ON Order VALUE string::uppercase(customer);


`overwrite=False` switches to `IF NOT EXISTS`, which never disturbs a definition that already
exists in the database:

In [4]:
print(Order.computed_field_ddl(overwrite=False)[0])

DEFINE FIELD IF NOT EXISTS subtotal ON Order VALUE unit_price * quantity;


## 4. Apply the schema

`define_computed_fields()` runs those statements and returns what it ran. It is **idempotent** --
the default `OVERWRITE` treats the model as the source of truth, so calling it at application
start-up converges the database onto your code (edit an expression, redeploy, done).

In [5]:
# Start from a clean table so this notebook is re-runnable.
client = await SurrealDBConnectionManager.get_client()
try:
    await client.query("REMOVE TABLE Order;", {})
except Exception:
    pass

applied = await Order.define_computed_fields()
print(f"{len(applied)} definitions applied")

3 definitions applied


## 5. Write a record -- the server fills the computed fields

Note what is *not* here: we never assign `subtotal`, `total` or `label`. The ORM strips computed
fields from the write payload entirely, and reads back what the server computed.

In [6]:
order = Order(id="a1", customer="ada lovelace", unit_price=12.5, quantity=4)
print("before save:", order.model_dump())
print("payload actually sent:", order._write_payload())

await order.save()
print("after save :", order.model_dump())

before save: {'id': 'a1', 'customer': 'ada lovelace', 'unit_price': 12.5, 'quantity': 4, 'subtotal': None, 'total': None, 'label': None}
payload actually sent: {'customer': 'ada lovelace', 'unit_price': 12.5, 'quantity': 4}
after save : {'id': 'a1', 'customer': 'ada lovelace', 'unit_price': 12.5, 'quantity': 4, 'subtotal': 50.0, 'total': 57.0, 'label': 'ADA LOVELACE'}


## 6. Every write verb recomputes

`merge`, `update`, `upsert` and `patch` all re-trigger the expression -- there is no "remember to
recalculate" step anywhere in your code.

In [7]:
await order.merge(quantity=10)
print(f"after merge(quantity=10)  -> subtotal={order.subtotal}, total={order.total}")

order.unit_price = 20.0
await order.update()
await order.refresh()
print(f"after update(unit_price)  -> subtotal={order.subtotal}, total={order.total}")

order.quantity = 3
await order.upsert()
print(f"after upsert(quantity=3)  -> subtotal={order.subtotal}, total={order.total}")

await order.patch([{"op": "replace", "path": "/unit_price", "value": 5.0}])
print(f"after patch(unit_price=5) -> subtotal={order.subtotal}, total={order.total}")

after merge(quantity=10)  -> subtotal=125.0, total=144.0
after update(unit_price)  -> subtotal=200.0, total=230.0
after upsert(quantity=3)  -> subtotal=60.0, total=69.0
after patch(unit_price=5) -> subtotal=15.0, total=17.0


Computed fields are ordinary columns as far as querying goes -- filter and order on them freely:

In [8]:
await Order(id="a2", customer="grace hopper", unit_price=100.0, quantity=2).save()

expensive = await Order.objects().filter(subtotal__gte=100).exec()
print("subtotal >= 100:", [(o.id, o.subtotal) for o in expensive])

ranked = await Order.objects().order_by("-total").exec()
print("by total desc  :", [(o.id, o.total) for o in ranked])

subtotal >= 100: [('a2', 200.0)]
by total desc  : [('a2', 230.0), ('a1', 17.0)]


## 7. The field is server-owned

Because the value belongs to the server, writing to it is meaningless -- SurrealDB would discard
it. Rather than let that fail silently, the ORM raises:

In [9]:
for attempt, call in {
    "merge(subtotal=...)": lambda: order.merge(subtotal=999),
    "bulk_update(total=...)": lambda: Order.objects().filter(customer="ada lovelace").bulk_update(total=1),
    "atomic_increment('subtotal')": lambda: order.atomic_increment("subtotal", 5),
}.items():
    try:
        await call()
    except ValueError as e:
        print(f"{attempt:32} -> ValueError: {e}")

merge(subtotal=...)              -> ValueError: merge(): subtotal is a computed field on Order, set server-side by DEFINE FIELD … VALUE and not writable. Update the fields the expression reads instead.
bulk_update(total=...)           -> ValueError: bulk_update(): total is a computed field on Order, set server-side by DEFINE FIELD … VALUE and not writable. Update the fields the expression reads instead.
atomic_increment('subtotal')     -> ValueError: atomic operation: subtotal is a computed field on Order, set server-side by DEFINE FIELD … VALUE and not writable. Update the fields the expression reads instead.


And this is genuinely enforced by the database, not merely by the ORM. Here we bypass ORM-lite
completely and send a hostile value straight over the SDK -- the expression still wins:

In [10]:
await client.query(
    "CREATE Order:evil CONTENT "
    "{customer: 'mallory', unit_price: 1.0, quantity: 1, subtotal: 999999, label: 'HACKED'};",
    {},
)
row = await client.query("SELECT * FROM Order:evil;", {})
print("client asked for subtotal=999999, label='HACKED'")
print("server stored     subtotal=%s, label=%r" % (row[0]["subtotal"], row[0]["label"]))

client asked for subtotal=999999, label='HACKED'
server stored     subtotal=1.0, label='MALLORY'


## 8. Two caveats worth knowing

**Evaluation order is alphabetical, not declaration order.** SurrealDB applies computed fields
sorted by field name, so a field that reads another must sort *after* it. In this notebook
`subtotal` -> `total` works because "subtotal" < "total". Had they been named `z_sub` and
`a_total`, the write would fail with `Cannot perform multiplication with 'none'`.

**The expression is inlined into DDL and cannot reference bound parameters.** It is
developer-controlled text, exactly like `SurrealFunc` -- never build one from user input.

A third, milder one: before `define_computed_fields()` has ever run, a computed field is simply
never written, so it reads back as `None`:

In [11]:
class Draft(BaseSurrealModel):
    id: str
    name: str = ""
    shout: Computed[str] = Computed("string::uppercase(name)")


draft = await Draft(id="d1", name="not defined yet").save()
print("without define_computed_fields() ->", draft.shout)

await Draft.define_computed_fields()
draft2 = await Draft(id="d2", name="defined now").save()
print("after  define_computed_fields() ->", draft2.shout)

without define_computed_fields() -> None
after  define_computed_fields() -> DEFINED NOW


## 9. Cleanup

In [12]:
for table in ("Order", "Draft"):
    try:
        await client.query(f"REMOVE TABLE {table};", {})
    except Exception:
        pass

await SurrealDBConnectionManager.close_connection()
print("Cleaned up and disconnected.")

Cleaned up and disconnected.
